In [1]:
%reset -f

In [2]:
# Standard imports
import sys
sys.path.insert(0, '/home/ubuntu/prem')

import os
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
import geopandas as gpd
from shapely import wkt
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Modular imports
from utils import log_memory, log_df_memory, save_detection_results
from filters.preprocessing import (
    filter_by_lifetime,
    attach_zones_to_objects,
    apply_footpath_zone_filter,
    compute_polygon_orientation,
    filter_parallel_vehicles,
    filter_static_objects
)
from regions.oulu.zones import (
    get_crosswalk_zone, 
    get_footpath_zones, 
    get_near_miss_zones, 
    get_exclusion_zone,
    get_lane_zones
)
from ssm.utils import load_config, find_all_nearby_pairs, get_mdrac_pairs
from ssm.m_drac import ModifiedDRAC

[SSM] Numba enabled with 7 threads


In [4]:
# Configuration
START_DATE = "2025-08-22"
END_DATE = "2025-09-11"
DATA_DIR = "/home/ubuntu/data/uploads/oulu_data/objects/clean/objects/clean"
OUTPUT_DIR = "/home/ubuntu/prem/results"

config = load_config("/home/ubuntu/prem/config.yaml")

print("="*70)
print("OULU PEDESTRIAN CROSSING ANALYSIS")
print("="*70)
print(f"Date: {START_DATE} to {END_DATE}")
print("="*70)

OULU PEDESTRIAN CROSSING ANALYSIS
Date: 2025-08-22 to 2025-09-11


## 1. Data Loading

In [5]:
def load_oulu_data(data_dir, start_date, end_date):
    """
    Load Oulu data from hourly parquet folders.
    
    Data structure: YYYY-MM-DD-HH/YYYY-MM-DD-HH-MM.parquet
    """
    dfs = []
    
    # List all folders
    folders = sorted([f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))])
    
    for folder in tqdm(folders, desc="Loading data"):
        # Extract date from folder name (YYYY-MM-DD-HH)
        try:
            folder_date = folder[:10]  # YYYY-MM-DD
            
            # Check if within date range
            if folder_date < start_date or folder_date > end_date:
                continue
            
            folder_path = os.path.join(data_dir, folder)
            
            # Load all parquet files in this hour folder
            df_hour = pd.read_parquet(folder_path)
            dfs.append(df_hour)
            
        except Exception as e:
            print(f"Error loading {folder}: {e}")
            continue
    
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        print(f"\n✓ Loaded {len(df):,} records from {len(dfs)} hour folders")
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

In [6]:
print("\nLoading Oulu data...")
log_memory("Before loading")

df = load_oulu_data(DATA_DIR, START_DATE, END_DATE)

log_df_memory(df, "Loaded data")
df.reset_index(drop=True, inplace=True)
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")


Loading Oulu data...
[MEMORY] Before loading: 194.7 MB


Loading data: 100%|██████████| 617/617 [00:06<00:00, 100.24it/s]



✓ Loaded 43,174,533 records from 481 hour folders
[DF MEMORY] Loaded data: 2635.2 MB (43,174,533 rows)

Date range: 2025-08-22 23:45:00.020003796 to 2025-09-11 23:59:59.922009945


## 2. Lifetime Filtering

In [7]:
print("\n" + "="*70)
print("Lifetime Filtering")
print("="*70)

df = filter_by_lifetime(df, config['preprocessing']['lifetime_filter']['min_lifespan'])
log_df_memory(df, "After lifetime filter")


Lifetime Filtering
[lifespan filter] Removed 280,488 short-lived IDs
  Before: 465,528 IDs (43,174,533 rows)
  After: 185,040 IDs (28,968,836 rows)
[DF MEMORY] After lifetime filter: 1989.1 MB (28,968,836 rows)


np.float64(1989.1321105957031)

## 3. Footpath Zone Filtering

In [8]:
print("\n" + "="*70)
print("Footpath Zone Filtering")
print("="*70)

footpath_zones = get_footpath_zones()
zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

print(f"Attaching footpath zones to {len(df):,} rows...")
log_memory("Before footpath zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After footpath zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

df = apply_footpath_zone_filter(df)
df = df.drop(columns=['zone'], errors='ignore')
df.reset_index(drop=True, inplace=True)
gc.collect()
log_memory("After footpath filter")


Footpath Zone Filtering
Attaching footpath zones to 28,968,836 rows...
[MEMORY] Before footpath zones: 5030.6 MB


Zone assignment batches: 100%|██████████| 290/290 [00:35<00:00,  8.10it/s]


[MEMORY] After footpath zones: 6998.2 MB
✓ Zones attached! Total rows: 28,968,836
[footpath zone] Removed 25501 objects
[MEMORY] After footpath filter: 6630.7 MB


6630.6796875

## 4. Crosswalk Zone Filtering

In [9]:
print("\n" + "="*70)
print("Crosswalk Zone Filtering (Remove Parallel Cars)")
print("="*70)

crosswalk_zone = get_crosswalk_zone()
zones_df = pd.DataFrame([crosswalk_zone])
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

print(f"Crosswalk orientation: {gdf_zones['orientation_deg'].iloc[0]:.2f}°")
print(f"Attaching crosswalk zone to {len(df):,} rows...")
log_memory("Before crosswalk zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After crosswalk zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

# Filter parallel vehicles (cars moving along crosswalk, not crossing)
removed_ids_global = []
df_in_zones = df[df['zone'].notnull()].copy()

if len(df_in_zones) > 0:
    orientation = gdf_zones['orientation_deg'].iloc[0]
    parallel_ids, _ = filter_parallel_vehicles(df_in_zones, orientation, threshold=4.0)
    removed_ids_global.extend(parallel_ids)
    df = df[~df['id'].isin(removed_ids_global)]
    print(f"[crosswalk] Removed {len(removed_ids_global):,} parallel vehicles")

df = df.drop(columns=['zone'], errors='ignore')
df.reset_index(drop=True, inplace=True)
gc.collect()
log_memory("After crosswalk filter")


Crosswalk Zone Filtering (Remove Parallel Cars)
Crosswalk orientation: -88.57°
Attaching crosswalk zone to 26,168,088 rows...
[MEMORY] Before crosswalk zones: 6630.8 MB


Zone assignment batches: 100%|██████████| 262/262 [00:32<00:00,  8.04it/s]


[MEMORY] After crosswalk zones: 6630.9 MB
✓ Zones attached! Total rows: 26,168,088
[crosswalk] Removed 25 parallel vehicles
[MEMORY] After crosswalk filter: 6601.8 MB


6601.7734375

## 5. Static Object Removal

In [10]:
print("\n" + "="*70)
print("Static Object Removal")
print("="*70)

df = filter_static_objects(df, 
    static_threshold=config['preprocessing']['static_filter']['min_speed'],
    static_ratio_min=0.8)

log_df_memory(df, "After static filter")


Static Object Removal
[static filter] Found 4,582 static objects
[static filter] Removed 4,582 static objects
  Before: 159,514 IDs (26,164,522 rows)
  After: 154,932 IDs (22,477,205 rows)
[DF MEMORY] After static filter: 1543.4 MB (22,477,205 rows)


np.float64(1543.3871841430664)

## 6. Exclusion Zone Filtering

In [11]:
print("\n" + "="*70)
print("Exclusion Zone Filtering")
print("="*70)

exclusion_zone = get_exclusion_zone()
exclusion_poly = wkt.loads(exclusion_zone["vertices"])

# VECTORIZED APPROACH - Much faster than df.apply()
from shapely.geometry import Point

# Create GeoDataFrame with point geometries
gdf_temp = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['pos_x'], df['pos_y'])
)

# Vectorized contains check (much faster than row-by-row apply)
df['in_exclusion_zone'] = gdf_temp.geometry.within(exclusion_poly)

removed = df['in_exclusion_zone'].sum()

df = df[~df['in_exclusion_zone']].copy()
df.drop(columns=['in_exclusion_zone'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"[exclusion zone] Removed {removed:,} observations")
log_df_memory(df, "After exclusion filter")


Exclusion Zone Filtering


[exclusion zone] Removed 4,503 observations
[DF MEMORY] After exclusion filter: 1371.6 MB (22,472,702 rows)


np.float64(1371.625)

## 7. Near-Miss Zone Assignment

In [12]:
print("\n" + "="*70)
print("Near-Miss Zone Assignment")
print("="*70)

near_miss_zones = get_near_miss_zones()
zones_df = pd.DataFrame(near_miss_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

print(f"Attaching near-miss zones to {len(df):,} rows...")
log_memory("Before near-miss zones")

df = attach_zones_to_objects(df, gdf_zones, how="inner", batch_size=100000)

log_memory("After near-miss zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

print("\nZone distribution:")
print(df['zone'].value_counts())

df_analysis = df.copy()
log_df_memory(df_analysis, "Analysis-ready data")


Near-Miss Zone Assignment
Attaching near-miss zones to 22,472,702 rows...
[MEMORY] Before near-miss zones: 11040.5 MB


Zone assignment batches: 100%|██████████| 225/225 [01:07<00:00,  3.32it/s]


[MEMORY] After near-miss zones: 10369.4 MB
✓ Zones attached! Total rows: 4,704,167

Zone distribution:
zone
1002    2611333
1001    2092834
Name: count, dtype: int64
[DF MEMORY] Analysis-ready data: 291.6 MB (4,704,167 rows)


np.float64(291.60615253448486)

## 8. M-DRAC Detection

In [13]:
print("\n" + "="*70)
print("M-DRAC Detection")
print("="*70)

# Generate base pairs
print("\nGenerating nearby pairs...")
log_memory("Before pair generation")

base_pairs = find_all_nearby_pairs(df_analysis, config)

print(f"✓ Generated {len(base_pairs):,} base pairs")
log_memory("After pair generation")

# Filter pairs for M-DRAC 
print("\nFiltering pairs for M-DRAC...")
mdrac_pairs = get_mdrac_pairs(base_pairs, config, skip_pair_generation=True)
print(f"✓ M-DRAC pairs after filtering: {len(mdrac_pairs):,}")

# Detect conflicts from filtered pairs
print("\nDetecting M-DRAC conflicts...")
mdrac_detector = ModifiedDRAC(config)
mdrac_conflicts = mdrac_detector.detect(mdrac_pairs, is_pairs_data=True)

print(f"\n{'='*70}")
print(f"M-DRAC Conflicts: {len(mdrac_conflicts):,}")
print(f"{'='*70}")


M-DRAC Detection

Generating nearby pairs...
[MEMORY] Before pair generation: 10620.6 MB
  Filtered vehicles: 1,227,630
  Generating pairs (max_distance=8.0m)...
  Processing 977,911 timestamps (batch_size=5,000)


  Pair generation: 100%|██████████| 196/196 [00:03<00:00, 49.62it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 138,800 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 138,800 base pairs
[MEMORY] After pair generation: 10672.1 MB

Filtering pairs for M-DRAC...
 Approaching pairs: 53,032
  Zone filter (same lane): 47,370 pairs (filtered 5,662 different-lane)
  Lateral filter (<= 3.0m): 12,385 pairs (filtered 34,985 not aligned)
  ✓ Total filtered: 40,647 pairs | Remaining: 12,385 pairs
  Speed diff > 0.5: 3,908
  Final MDRAC pairs: 26
✓ M-DRAC pairs after filtering: 26

Detecting M-DRAC conflicts...
 Approaching pairs: 26
  Zone filter (same lane): 26 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 26 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 26 pairs
  Speed diff > 0.5: 26
  Final MDRAC pairs: 26

M-DRAC Conflicts: 0


In [21]:
# Save M-DRAC results
if len(mdrac_conflicts) > 0:
    mdrac_path = save_detection_results(mdrac_conflicts, OUTPUT_DIR, 'mdrac', 'oulu', START_DATE)
    print(f"✓ Saved to {mdrac_path}")
    
    # Show sample (MDRAC output uses uppercase column names)
    print("\nSample conflicts:")
    print(mdrac_conflicts[['timestamp', 'id1', 'id2', 'interaction', 'MDRAC', 'TTC', 'dist']].head(10))
else:
    print("No conflicts detected.")

No conflicts detected.


## Summary

In [22]:
print("\n" + "="*70)
print("OULU ANALYSIS COMPLETE")
print("="*70)
print(f"Date: {START_DATE} to {END_DATE}")
print(f"Final objects in near-miss zones: {len(df_analysis):,}")
print(f"M-DRAC conflicts detected: {len(mdrac_conflicts):,}")
print("="*70)


OULU ANALYSIS COMPLETE
Date: 2025-08-22 to 2025-09-11
Final objects in near-miss zones: 4,704,167
M-DRAC conflicts detected: 0


---

# PART 2: LANE-BASED NEAR-MISS DETECTION

This section analyzes near-misses within individual lanes (R-A, R-B, R-C).
Each lane is processed separately to detect intra-lane conflicts only.

## 9. Load Data for Lane Analysis

In [23]:
print("\n" + "="*70)
print("LOADING DATA FOR LANE ANALYSIS")
print("="*70)

# Reload fresh data (to avoid using crossing-filtered data)
print("\nReloading Oulu data...")
log_memory("Before lane data loading")

df_lanes = load_oulu_data(DATA_DIR, START_DATE, END_DATE)

log_df_memory(df_lanes, "Loaded lane data")
df_lanes.reset_index(drop=True, inplace=True)
print(f"Date range: {df_lanes['timestamp'].min()} to {df_lanes['timestamp'].max()}")


LOADING DATA FOR LANE ANALYSIS

Reloading Oulu data...
[MEMORY] Before lane data loading: 14379.8 MB


Loading data: 100%|██████████| 617/617 [00:01<00:00, 341.78it/s] 



✓ Loaded 43,174,533 records from 481 hour folders
[DF MEMORY] Loaded lane data: 2635.2 MB (43,174,533 rows)
Date range: 2025-08-22 23:45:00.020003796 to 2025-09-11 23:59:59.922009945


## 10. Apply Basic Filters for Lane Data

In [24]:
print("\n" + "="*70)
print("APPLYING FILTERS TO LANE DATA")
print("="*70)

# Lifetime filter
print("\n[1/3] Lifetime filtering...")
df_lanes = filter_by_lifetime(df_lanes, config['preprocessing']['lifetime_filter']['min_lifespan'])
log_df_memory(df_lanes, "After lifetime filter")

# Static object removal
print("\n[2/3] Removing static objects...")
df_lanes = filter_static_objects(df_lanes, 
    static_threshold=config['preprocessing']['static_filter']['min_speed'],
    static_ratio_min=0.8)
log_df_memory(df_lanes, "After static filter")

# Exclusion zone filter
print("\n[3/3] Applying exclusion zone...")
exclusion_zone = get_exclusion_zone()
exclusion_poly = wkt.loads(exclusion_zone["vertices"])

gdf_temp = gpd.GeoDataFrame(
    df_lanes,
    geometry=gpd.points_from_xy(df_lanes['pos_x'], df_lanes['pos_y'])
)
df_lanes['in_exclusion_zone'] = gdf_temp.geometry.within(exclusion_poly)
removed = df_lanes['in_exclusion_zone'].sum()
df_lanes = df_lanes[~df_lanes['in_exclusion_zone']].copy()
df_lanes.drop(columns=['in_exclusion_zone'], inplace=True)
df_lanes.reset_index(drop=True, inplace=True)

print(f"[exclusion zone] Removed {removed:,} observations")
log_df_memory(df_lanes, "After all filters")
gc.collect()


APPLYING FILTERS TO LANE DATA

[1/3] Lifetime filtering...


[lifespan filter] Removed 280,488 short-lived IDs
  Before: 465,528 IDs (43,174,533 rows)
  After: 185,040 IDs (28,968,836 rows)
[DF MEMORY] After lifetime filter: 1989.1 MB (28,968,836 rows)

[2/3] Removing static objects...
[static filter] Found 4,639 static objects
[static filter] Removed 4,639 static objects
  Before: 185,040 IDs (28,968,836 rows)
  After: 180,401 IDs (25,219,179 rows)
[DF MEMORY] After static filter: 1731.7 MB (25,219,179 rows)

[3/3] Applying exclusion zone...
[exclusion zone] Removed 4,793 observations
[DF MEMORY] After all filters: 1539.0 MB (25,214,386 rows)


0

## 11. Attach Lane Zones

In [25]:
print("\n" + "="*70)
print("ATTACHING LANE ZONES")
print("="*70)

lane_zones = get_lane_zones()
zones_df_lanes = pd.DataFrame(lane_zones)
zones_df_lanes["geometry"] = zones_df_lanes["vertices"].apply(wkt.loads)
gdf_lane_zones = gpd.GeoDataFrame(zones_df_lanes, geometry="geometry")

print(f"\nLane zones loaded:")
for idx, row in gdf_lane_zones.iterrows():
    print(f"  - {row['name']} (ID: {row['id']})")

print(f"\nAttaching lane zones to {len(df_lanes):,} rows...")
log_memory("Before lane zone attachment")

df_lanes = attach_zones_to_objects(df_lanes, gdf_lane_zones, how="inner", batch_size=100000)

log_memory("After lane zone attachment")
print(f"✓ Zones attached! Total rows in lanes: {len(df_lanes):,}")

print("\nLane distribution:")
lane_dist = df_lanes['zone'].value_counts()
for lane, count in lane_dist.items():
    print(f"  {lane}: {count:,} objects ({count/len(df_lanes)*100:.1f}%)")

df_lanes.reset_index(drop=True, inplace=True)
gc.collect()


ATTACHING LANE ZONES

Lane zones loaded:
  - R-A (ID: 2001)
  - R-B (ID: 2002)
  - R-C (ID: 2003)

Attaching lane zones to 25,214,386 rows...
[MEMORY] Before lane zone attachment: 15400.4 MB


Zone assignment batches: 100%|██████████| 253/253 [01:28<00:00,  2.87it/s]


[MEMORY] After lane zone attachment: 15400.8 MB
✓ Zones attached! Total rows in lanes: 9,765,113

Lane distribution:
  2002: 4,998,345 objects (51.2%)
  2001: 4,290,365 objects (43.9%)
  2003: 476,403 objects (4.9%)


0

## 12. Separate Lane Processing - MDRAC Detection

Process each lane independently to detect intra-lane conflicts only.

In [26]:
print("\n" + "="*70)
print("SEPARATE LANE MDRAC DETECTION")
print("="*70)

all_lane_conflicts = []
lane_stats = {}

# Get unique lanes
unique_lanes = df_lanes['zone'].unique()

for lane_name in unique_lanes:
    print(f"\n{'='*70}")
    print(f"Processing Lane: {lane_name}")
    print(f"{'='*70}")
    
    # Filter to this lane only
    df_lane = df_lanes[df_lanes['zone'] == lane_name].copy()
    print(f"Objects in {lane_name}: {len(df_lane):,}")
    
    if len(df_lane) < 2:
        print(f"⚠ Skipping {lane_name} - insufficient objects")
        lane_stats[lane_name] = {'objects': len(df_lane), 'conflicts': 0}
        continue
    
    # Generate pairs within this lane
    print(f"Generating nearby pairs for {lane_name}...")
    lane_pairs = find_all_nearby_pairs(df_lane, config)
    print(f"✓ Generated {len(lane_pairs):,} base pairs")
    
    if len(lane_pairs) == 0:
        print(f"⚠ No pairs found in {lane_name}")
        lane_stats[lane_name] = {'objects': len(df_lane), 'conflicts': 0}
        continue
    
    # Filter for MDRAC
    print(f"Filtering pairs for M-DRAC...")
    mdrac_lane_pairs = get_mdrac_pairs(lane_pairs, config, skip_pair_generation=True)
    print(f"✓ M-DRAC pairs: {len(mdrac_lane_pairs):,}")
    
    if len(mdrac_lane_pairs) == 0:
        print(f"⚠ No MDRAC pairs in {lane_name}")
        lane_stats[lane_name] = {'objects': len(df_lane), 'conflicts': 0}
        continue
    
    # Detect conflicts
    print(f"Detecting M-DRAC conflicts in {lane_name}...")
    mdrac_detector = ModifiedDRAC(config)
    lane_conflicts = mdrac_detector.detect(mdrac_lane_pairs, is_pairs_data=True)
    
    # Add lane identifier
    if len(lane_conflicts) > 0:
        lane_conflicts['lane'] = lane_name
        all_lane_conflicts.append(lane_conflicts)
        print(f"✓ Detected {len(lane_conflicts):,} conflicts in {lane_name}")
    else:
        print(f"✓ No conflicts detected in {lane_name}")
    
    lane_stats[lane_name] = {
        'objects': len(df_lane),
        'conflicts': len(lane_conflicts) if len(lane_conflicts) > 0 else 0
    }
    
    # Clear memory
    del df_lane, lane_pairs, mdrac_lane_pairs
    if len(lane_conflicts) > 0:
        del lane_conflicts
    gc.collect()

# Combine all lane conflicts
if all_lane_conflicts:
    df_lane_conflicts = pd.concat(all_lane_conflicts, ignore_index=True)
    print(f"\n{'='*70}")
    print(f"TOTAL LANE CONFLICTS: {len(df_lane_conflicts):,}")
    print(f"{'='*70}")
else:
    df_lane_conflicts = pd.DataFrame()
    print(f"\n{'='*70}")
    print("NO LANE CONFLICTS DETECTED")
    print(f"{'='*70}")


SEPARATE LANE MDRAC DETECTION

Processing Lane: 2001
Objects in 2001: 4,290,365
Generating nearby pairs for 2001...
  Filtered vehicles: 1,405,965
  Generating pairs (max_distance=8.0m)...
  Processing 892,251 timestamps (batch_size=5,000)


  Pair generation: 100%|██████████| 179/179 [00:03<00:00, 48.58it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 168,879 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 168,879 base pairs
Filtering pairs for M-DRAC...
 Approaching pairs: 77,220
  Zone filter (same lane): 77,220 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 24,296 pairs (filtered 52,924 not aligned)
  ✓ Total filtered: 52,924 pairs | Remaining: 24,296 pairs
  Speed diff > 0.5: 10,751
  Final MDRAC pairs: 883
✓ M-DRAC pairs: 883
Detecting M-DRAC conflicts in 2001...
 Approaching pairs: 883
  Zone filter (same lane): 883 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 883 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 883 pairs
  Speed diff > 0.5: 883
  Final MDRAC pairs: 883
✓ Detected 28 conflicts in 2001

Processing Lane: 2002
Objects in 2002: 4,998,345
Generating nearby pairs for 2002...
  Filtered vehicles: 2,149,217
  Generating pairs (max_distance=8.0m)...
  Processin

  Pair generation: 100%|██████████| 294/294 [00:08<00:00, 32.95it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 181,772 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 181,772 base pairs
Filtering pairs for M-DRAC...
 Approaching pairs: 77,990
  Zone filter (same lane): 77,990 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 27,016 pairs (filtered 50,974 not aligned)
  ✓ Total filtered: 50,974 pairs | Remaining: 27,016 pairs
  Speed diff > 0.5: 10,651
  Final MDRAC pairs: 1,567
✓ M-DRAC pairs: 1,567
Detecting M-DRAC conflicts in 2002...
 Approaching pairs: 1,567
  Zone filter (same lane): 1,567 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 1,567 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 1,567 pairs
  Speed diff > 0.5: 1,567
  Final MDRAC pairs: 1,567
✓ Detected 52 conflicts in 2002

Processing Lane: 2003
Objects in 2003: 476,403
Generating nearby pairs for 2003...
  Filtered vehicles: 41,268
  Generating pairs (max_distance=8.0m)...


  Pair generation: 100%|██████████| 9/9 [00:00<00:00, 339.04it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 130 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 130 base pairs
Filtering pairs for M-DRAC...
 Approaching pairs: 84
  Zone filter (same lane): 84 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 84 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 84 pairs
  Speed diff > 0.5: 32
  Final MDRAC pairs: 0
✓ M-DRAC pairs: 0
⚠ No MDRAC pairs in 2003

TOTAL LANE CONFLICTS: 80


## 13. Lane Analysis Results & Verification

In [27]:
print("\n" + "="*70)
print("LANE ANALYSIS STATISTICS")
print("="*70)

# Per-lane statistics
print("\nPer-Lane Statistics:")
print("-" * 70)
print(f"{'Lane':<10} {'Objects':<15} {'Conflicts':<15} {'Conflict Rate':<15}")
print("-" * 70)

for lane_name, stats in lane_stats.items():
    conflict_rate = (stats['conflicts'] / stats['objects'] * 100) if stats['objects'] > 0 else 0
    print(f"{lane_name:<10} {stats['objects']:<15,} {stats['conflicts']:<15,} {conflict_rate:<15.3f}%")

print("-" * 70)
total_lane_objects = sum(s['objects'] for s in lane_stats.values())
total_lane_conflicts = sum(s['conflicts'] for s in lane_stats.values())
overall_rate = (total_lane_conflicts / total_lane_objects * 100) if total_lane_objects > 0 else 0
print(f"{'TOTAL':<10} {total_lane_objects:<15,} {total_lane_conflicts:<15,} {overall_rate:<15.3f}%")
print("="*70)

# Conflict distribution by lane
if len(df_lane_conflicts) > 0:
    print("\n" + "="*70)
    print("CONFLICT DISTRIBUTION BY LANE")
    print("="*70)
    lane_conflict_counts = df_lane_conflicts['lane'].value_counts()
    for lane, count in lane_conflict_counts.items():
        print(f"  {lane}: {count:,} conflicts ({count/len(df_lane_conflicts)*100:.1f}%)")
    
    # Verify no cross-lane conflicts
    print("\n" + "="*70)
    print("VERIFICATION: Checking for Cross-Lane Conflicts")
    print("="*70)
    
    # Check if both objects in each conflict are from the same lane
    # This requires checking if id1 and id2 both belong to the same lane
    print("✓ All conflicts are intra-lane (by design - separate processing)")
    print("✓ No cross-lane conflicts detected")
    
    # Sample conflicts per lane (using correct column names from MDRAC output)
    print("\n" + "="*70)
    print("SAMPLE CONFLICTS PER LANE (Top 5 each)")
    print("="*70)
    
    for lane_name in unique_lanes:
        lane_sample = df_lane_conflicts[df_lane_conflicts['lane'] == lane_name].head(5)
        if len(lane_sample) > 0:
            print(f"\n{lane_name}:")
            # MDRAC detector outputs: MDRAC, TTC (uppercase), interaction, dist (not distance)
            print(lane_sample[['timestamp', 'id1', 'id2', 'interaction', 'MDRAC', 'TTC', 'dist']].to_string(index=False))
else:
    print("\n⚠ No conflicts to analyze")


LANE ANALYSIS STATISTICS

Per-Lane Statistics:
----------------------------------------------------------------------
Lane       Objects         Conflicts       Conflict Rate  
----------------------------------------------------------------------
2001       4,290,365       28              0.001          %
2002       4,998,345       52              0.001          %
2003       476,403         0               0.000          %
----------------------------------------------------------------------
TOTAL      9,765,113       80              0.001          %

CONFLICT DISTRIBUTION BY LANE
  2002: 52 conflicts (65.0%)
  2001: 28 conflicts (35.0%)

VERIFICATION: Checking for Cross-Lane Conflicts
✓ All conflicts are intra-lane (by design - separate processing)
✓ No cross-lane conflicts detected

SAMPLE CONFLICTS PER LANE (Top 5 each)

2001:
                    timestamp    id1    id2 interaction     MDRAC      TTC     dist
2025-09-09 13:30:34.408506155 188172 188264   car_v_car  7.576981 1.140

In [28]:
# Save lane-based results
if len(df_lane_conflicts) > 0:
    lane_output_dir = os.path.join(OUTPUT_DIR, "oulu", "mdrac_lanes")
    os.makedirs(lane_output_dir, exist_ok=True)
    
    # Save with date in filename
    output_filename = f"mdrac_lanes_{START_DATE}_to_{END_DATE}.csv"
    output_path = os.path.join(lane_output_dir, output_filename)
    
    df_lane_conflicts.to_csv(output_path, index=False)
    print(f"\n✓ Saved lane conflicts to: {output_path}")
    
    # Also save lane statistics
    stats_df = pd.DataFrame([
        {'lane': lane, 'objects': stats['objects'], 'conflicts': stats['conflicts']}
        for lane, stats in lane_stats.items()
    ])
    stats_path = os.path.join(lane_output_dir, f"lane_stats_{START_DATE}_to_{END_DATE}.csv")
    stats_df.to_csv(stats_path, index=False)
    print(f"✓ Saved lane statistics to: {stats_path}")
else:
    print("\n⚠ No lane conflicts to save")


✓ Saved lane conflicts to: /home/ubuntu/prem/results/oulu/mdrac_lanes/mdrac_lanes_2025-08-22_to_2025-09-11.csv
✓ Saved lane statistics to: /home/ubuntu/prem/results/oulu/mdrac_lanes/lane_stats_2025-08-22_to_2025-09-11.csv


## Final Summary - Both Analyses

In [29]:
print("\n" + "="*70)
print("OULU REGION - COMPLETE ANALYSIS SUMMARY")
print("="*70)
print(f"Date Range: {START_DATE} to {END_DATE}")
print("="*70)

print("\n[PART 1] PEDESTRIAN CROSSING ANALYSIS")
print("-" * 70)
print(f"  Final objects in near-miss zones: {len(df_analysis):,}")
print(f"  M-DRAC conflicts detected: {len(mdrac_conflicts):,}")

print("\n[PART 2] LANE-BASED ANALYSIS")
print("-" * 70)
print(f"  Lanes analyzed: {len(unique_lanes)}")
print(f"  Total objects in lanes: {total_lane_objects:,}")
print(f"  Total intra-lane conflicts: {total_lane_conflicts:,}")

print("\n" + "="*70)
print("VERIFICATION SUMMARY")
print("="*70)
print("✓ Pedestrian crossing: Zone-specific near-miss detection")
print("✓ Lane analysis: Separate intra-lane processing (no cross-lane conflicts)")
print("✓ Both analyses use independent data pipelines")
print("="*70)


OULU REGION - COMPLETE ANALYSIS SUMMARY
Date Range: 2025-08-22 to 2025-09-11

[PART 1] PEDESTRIAN CROSSING ANALYSIS
----------------------------------------------------------------------
  Final objects in near-miss zones: 4,704,167
  M-DRAC conflicts detected: 0

[PART 2] LANE-BASED ANALYSIS
----------------------------------------------------------------------
  Lanes analyzed: 3
  Total objects in lanes: 9,765,113
  Total intra-lane conflicts: 80

VERIFICATION SUMMARY
✓ Pedestrian crossing: Zone-specific near-miss detection
✓ Lane analysis: Separate intra-lane processing (no cross-lane conflicts)
✓ Both analyses use independent data pipelines
